# 05 — Ablation Analysis**Input:** `../data/processed/pm_day_features.csv`, `../data/processed/embeddings.npy`**Output:** `../outputs/tables/nested_ablation_*.csv`, `../outputs/tables/state_trait_glm.csv`**Description:**- Nested ablation: content vs engagement vs instability (mechanism adjudication)- Incremental utility: does text add value beyond numeric crisis ratings?- CV-safe instability (centroids computed from training folds only)- State vs trait decomposition via cluster-robust GLM

In [ ]:
import osimport numpy as npimport pandas as pdfrom sklearn.model_selection import GroupKFoldfrom sklearn.decomposition import PCAfrom sklearn.preprocessing import StandardScalerfrom sklearn.linear_model import LogisticRegressionfrom sklearn.pipeline import Pipelinefrom sklearn.metrics import roc_auc_score, average_precision_scoreimport statsmodels.api as sm# =========================# CONFIG# =========================DATA_PATH = os.path.join("..", "data", "processed", "pm_day_features.csv")EMBED_PATH = os.path.join("..", "data", "processed", "embeddings.npy")OUT_DIR = os.path.join("..", "outputs", "tables")os.makedirs(OUT_DIR, exist_ok=True)PID_COL = "expiwell_id_clean"TEXT_COL = "pm_day_text"CRISIS_COL = "crisis_PM_from_full"N_PCS = 20RANDOM_SEED = 7np.random.seed(RANDOM_SEED)

In [ ]:
# =========================# LOAD# =========================pm_day = pd.read_csv(DATA_PATH)X_text = np.load(EMBED_PATH)assert X_text.shape[0] == len(pm_day)print("Loaded data:", pm_day.shape)

In [ ]:
# =========================# PREPARE FEATURE BLOCKS# =========================# EngagementX_eng = pm_day[["log1p_wc", "wc_le1"]].values.astype(float)# Numeric crisis predictorX_crisis = pm_day[[CRISIS_COL]].astype(float).values# Groups and filteringgroups = pm_day[PID_COL].astype(str).valuesmask_ok = (    np.isfinite(X_crisis).all(axis=1) &    np.isfinite(X_eng).all(axis=1))print(f"Rows retained: {mask_ok.sum()} / {len(pm_day)}")

In [ ]:
# =========================# HELPERS# =========================def l2_normalize_rows(X, eps=1e-12):    n = np.linalg.norm(X, axis=1, keepdims=True)    return X / np.maximum(n, eps)def cosine_distance_to_centroid(X_unit, centroid):    sim = X_unit @ centroid    return 1.0 - simdef fit_predict_logit(Xtr, Xte, ytr, C=1.0, random_state=0):    pipe = Pipeline([        ("scaler", StandardScaler()),        ("logit", LogisticRegression(            penalty="l2", C=C, solver="liblinear",            max_iter=5000, random_state=random_state        ))    ])    pipe.fit(Xtr, ytr)    return pipe.predict_proba(Xte)[:, 1]

In [ ]:
# =========================# NESTED ABLATION (with CV-safe instability)# =========================def run_nested_ablation(X_text, X_eng, X_crisis, y, groups, n_pcs=20):    gkf = GroupKFold(n_splits=5)    X_text_norm = l2_normalize_rows(X_text)    oof = {k: np.full(len(y), np.nan) for k in [        "C_content",        "CE_content_eng",        "CEI_content_eng_instab",        "N_numeric",        "NC_numeric_content",        "NCE_numeric_content_eng",        "NCEI_numeric_content_eng_instab",    ]}    for tr, te in gkf.split(X_text, y, groups):        # PCA on content (inside fold)        pca = PCA(n_components=min(n_pcs, X_text.shape[1]), random_state=RANDOM_SEED)        Ttr = pca.fit_transform(X_text[tr])        Tte = pca.transform(X_text[te])        # CV-safe instability        tr_idx, te_idx = np.array(tr), np.array(te)        train_groups = groups[tr_idx]        global_centroid = X_text_norm[tr_idx].mean(axis=0)        global_centroid /= max(np.linalg.norm(global_centroid), 1e-12)        pid_to_centroid = {}        for pid in np.unique(train_groups):            rows = X_text_norm[tr_idx[train_groups == pid]]            c = rows.mean(axis=0)            c /= max(np.linalg.norm(c), 1e-12)            pid_to_centroid[pid] = c        instab_tr = np.array([1.0 - X_text_norm[idx] @ pid_to_centroid.get(groups[idx], global_centroid)                              for idx in tr_idx]).reshape(-1, 1)        instab_te = np.array([1.0 - X_text_norm[idx] @ pid_to_centroid.get(groups[idx], global_centroid)                              for idx in te_idx]).reshape(-1, 1)        # Mechanism tests (no crisis baseline)        oof["C_content"][te] = fit_predict_logit(Ttr, Tte, y[tr])        oof["CE_content_eng"][te] = fit_predict_logit(            np.hstack([Ttr, X_eng[tr]]), np.hstack([Tte, X_eng[te]]), y[tr])        oof["CEI_content_eng_instab"][te] = fit_predict_logit(            np.hstack([Ttr, X_eng[tr], instab_tr]), np.hstack([Tte, X_eng[te], instab_te]), y[tr])        # Incremental utility tests (with crisis baseline)        oof["N_numeric"][te] = fit_predict_logit(X_crisis[tr], X_crisis[te], y[tr])        oof["NC_numeric_content"][te] = fit_predict_logit(            np.hstack([X_crisis[tr], Ttr]), np.hstack([X_crisis[te], Tte]), y[tr])        oof["NCE_numeric_content_eng"][te] = fit_predict_logit(            np.hstack([X_crisis[tr], Ttr, X_eng[tr]]),            np.hstack([X_crisis[te], Tte, X_eng[te]]), y[tr])        oof["NCEI_numeric_content_eng_instab"][te] = fit_predict_logit(            np.hstack([X_crisis[tr], Ttr, X_eng[tr], instab_tr]),            np.hstack([X_crisis[te], Tte, X_eng[te], instab_te]), y[tr])    rows = []    for name, p in oof.items():        rows.append({            "model": name,            "AUROC": roc_auc_score(y, p),            "AUPRC": average_precision_score(y, p)        })    return pd.DataFrame(rows).sort_values("model")

In [ ]:
# =========================# RUN ABLATION FOR EACH OUTCOME# =========================for outcome in ["high_any_item_eq3", "moderate_total_ge2", "any_risk_total_gt0"]:    y = pm_day[outcome].astype(int).values    # Apply missingness mask    y_m = y[mask_ok]    X_text_m = X_text[mask_ok]    X_eng_m = X_eng[mask_ok]    X_crisis_m = X_crisis[mask_ok]    groups_m = groups[mask_ok]    if y_m.sum() < 20:        print(f"Skipping {outcome}: too few positives")        continue    print(f"\n{'='*60}")    print(f"Outcome: {outcome} | N={len(y_m)} | pos={y_m.sum()} ({y_m.mean():.3f})")    print(f"{'='*60}")    results = run_nested_ablation(X_text_m, X_eng_m, X_crisis_m, y_m, groups_m, n_pcs=N_PCS)    print(results.to_string(index=False))    out_path = os.path.join(OUT_DIR, f"nested_ablation_{outcome}.csv")    results.to_csv(out_path, index=False)    print(f"Saved: {out_path}")

In [ ]:
# =========================# STATE vs TRAIT GLM (cluster-robust)# =========================N_PCS_CONTENT = 5OUTCOME_BIN = "high_any_item_eq3"MECH_COLS = ["log1p_wc", "wc_le1", "instability_cosdist"] + [f"PC{i}" for i in range(1, N_PCS_CONTENT + 1)]# Use within/between columns already computed in feature engineeringX_cols = []for c in MECH_COLS:    X_cols += [f"{c}_within", f"{c}_between"]# Filterdfm = pm_day.copy()y_glm = dfm[OUTCOME_BIN].astype(int).valuesmask_glm = np.isfinite(y_glm)for c in X_cols:    if c in dfm.columns:        mask_glm &= np.isfinite(dfm[c].values)dfm = dfm.loc[mask_glm].copy()y_glm = dfm[OUTCOME_BIN].astype(int).valuesX_glm = sm.add_constant(dfm[X_cols].astype(float), has_constant="add")groups_glm = dfm[PID_COL].astype(str).valuesglm = sm.GLM(y_glm, X_glm, family=sm.families.Binomial())res = glm.fit(cov_type="cluster", cov_kwds={"groups": groups_glm})print(f"\n=== STATE vs TRAIT (cluster-robust GLM) ===")print(f"Outcome: {OUTCOME_BIN} | N={len(dfm)} | participants={dfm[PID_COL].nunique()}")print(res.summary())# Save slide-friendly tablerows = []for c in MECH_COLS:    for part in ["within", "between"]:        name = f"{c}_{part}"        if name in res.params.index:            rows.append({                "feature": c, "component": part,                "beta": float(res.params[name]),                "SE": float(res.bse[name]),                "p": float(res.pvalues[name]),                "OR": float(np.exp(res.params[name])),            })tab = pd.DataFrame(rows)tab_path = os.path.join(OUT_DIR, "state_trait_glm.csv")tab.to_csv(tab_path, index=False)print(f"Saved: {tab_path}")